# Mini-Ablations Smoke Test (CPU)

**Purpose**: Tests all 7 augmentation configurations in one notebook

**Configurations**:
- **No Mosaic**: mosaic gives spatial diversity
- **No RandAugment**: Unpredictable transforms hurt clinical color
- **No Color Augmentations**: Color augmentations alter diagnostic features
- **Minimal (Mosaic and Flip Only)**: only the essential augmentations
- **No Translate**: Translate redundant with mosaic
- **Clinical Conservative**: Reduced safe augmentations match baseline
- **No Scale**: Scale redundant with mosaic

**Smoke Test Parameters**:
- Dataset: TINY (4 train images, 2 test images)
- Training: 1 epoch per config, batch=4 (CPU-compatible)
- Expected Runtime: ~21 minutes (7 configs × 3 min each)
- Purpose: Validate configurations before GPU deployment

**Reference**: `docs/BASELINE_VS_CLINICAL_AUGMENTATION_COMPARISON.md`. Not in the repo.


In [ ]:
# NumPy Version Locking
import numpy as np
import subprocess

default_numpy = np.__version__
print(f"Default NumPy: {default_numpy}")

subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
subprocess.run(
    ["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"],
    check=True
)

print(f"Locked NumPy: {np.__version__}")


In [ ]:
import sys
from pathlib import Path
from ultralytics import YOLO
import cv2
import json
import time

print(f"Python: {sys.version}")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")


In [ ]:
# Environment Setup
IN_KAGGLE = Path('/kaggle/input').exists()

if IN_KAGGLE:
    BASE_INPUT_PATH = Path('/kaggle/input')
    OUTPUT_PATH = Path('/kaggle/working')
    TINY_PATH = BASE_INPUT_PATH / 'tiny-wound-seg-smoke-test'
else:
    print("ERROR: Must run on Kaggle")
    sys.exit(1)

print(f"Output path: {OUTPUT_PATH}")
print(f"TINY path: {TINY_PATH}")


In [ ]:
# YOLO Configuration
config_content = f'''
path: {TINY_PATH}
train: train/images
val: test/images

names:
  0: background
  1: wound
'''

config_path = OUTPUT_PATH / 'wound_dataset.yaml'
config_path.write_text(config_content.strip())

print(f"YOLO config created: {config_path}")


In [ ]:
# Mini-Ablation Configurations
MINI_ABLATIONS = {
    "1_no_mosaic": {
        "name": "No Mosaic",
        "hypothesis": "Mosaic provides critical spatial diversity",
        "expected": "70-75% Dice (significant drop)",
        "params": {'mosaic': 0.0}
    },
    "7_no_randaugment": {
        "name": "No RandAugment",
        "hypothesis": "Unpredictable transforms hurt clinical color",
        "expected": "78-82% Dice (similar or better)",
        "params": {'auto_augment': None}
    },
    "2_no_color": {
        "name": "No Color Augmentations",
        "hypothesis": "Color augmentations alter diagnostic features",
        "expected": "78-82% Dice",
        "params": {'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'erasing': 0.0, 'auto_augment': None}
    },
    "4_minimal": {
        "name": "Minimal (Mosaic+Flip Only)",
        "hypothesis": "Only essential augmentations needed",
        "expected": "75-80% Dice",
        "params": {'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'mosaic': 1.0, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'erasing': 0.0, 'auto_augment': None}
    },
    "5_no_translate": {
        "name": "No Translate",
        "hypothesis": "Translate redundant with mosaic",
        "expected": "78-82% Dice",
        "params": {'translate': 0.0}
    },
    "6_clinical_conservative": {
        "name": "Clinical Conservative",
        "hypothesis": "Reduced safe augmentations match baseline",
        "expected": "78-82% Dice",
        "params": {'degrees': 0.0, 'translate': 0.1, 'scale': 0.3, 'flipud': 0.0, 'fliplr': 0.5, 'mosaic': 1.0, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.2, 'erasing': 0.0, 'auto_augment': None}
    },
    "3_no_scale": {
        "name": "No Scale",
        "hypothesis": "Scale redundant with mosaic",
        "expected": "78-82% Dice",
        "params": {'scale': 0.0}
    },
}

BASELINE_PARAMS = {'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'mosaic': 1.0, 'mixup': 0.0, 'copy_paste': 0.0, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'bgr': 0.0, 'erasing': 0.4, 'auto_augment': 'randaugment', 'close_mosaic': 10}

SMOKE_TEST_PARAMS = {'epochs': 1, 'batch': 4, 'imgsz': 512, 'patience': 10, 'seed': 42}

print(f"Testing {len(MINI_ABLATIONS)} mini-ablation configurations")
print(f"Smoke test: {SMOKE_TEST_PARAMS['epochs']} epoch, batch={SMOKE_TEST_PARAMS['batch']}")


In [ ]:
# Train All Mini-Ablations
results_summary = {}

for ablation_id, config in MINI_ABLATIONS.items():
    print("\n" + "=" * 80)
    print(f"MINI-ABLATION: {config['name']}")
    print("=" * 80)
    print(f"Hypothesis: {config['hypothesis']}")
    print(f"Expected: {config['expected']}")

    # Merge baseline params with ablation-specific overrides
    augmentation_params = BASELINE_PARAMS.copy()
    augmentation_params.update(config['params'])

    print("\nAugmentation Parameters:")
    for key, value in sorted(augmentation_params.items()):
        if value != BASELINE_PARAMS.get(key):
            print(f"  {key}: {value} ← CHANGED")
        else:
            print(f"  {key}: {value}")

    # Initialize model
    model = YOLO('yolov8n-seg.pt')

    # Train
    print(f"\nTraining...")
    start_time = time.time()

    try:
        results = model.train(
            data=str(config_path),
            batch=SMOKE_TEST_PARAMS['batch'],
            epochs=SMOKE_TEST_PARAMS['epochs'],
            imgsz=SMOKE_TEST_PARAMS['imgsz'],
            patience=SMOKE_TEST_PARAMS['patience'],
            seed=SMOKE_TEST_PARAMS['seed'],
            **augmentation_params,
            save=True,
            project=str(OUTPUT_PATH / "mini_ablations"),
            name=ablation_id,
            exist_ok=True,
            verbose=False
        )

        elapsed = time.time() - start_time
        status = "success"
        error_msg = None

        print(f"OK Training completed in {elapsed/60:.1f} minutes")

    except Exception as e:
        elapsed = time.time() - start_time
        status = "failed"
        error_msg = str(e)
        print(f"FAIL Training failed: {error_msg}")

    # Store results
    results_summary[ablation_id] = {
        "name": config['name'],
        "hypothesis": config['hypothesis'],
        "expected": config['expected'],
        "augmentation_params": augmentation_params,
        "status": status,
        "error": error_msg,
        "training_time_seconds": elapsed
    }

print("\n" + "=" * 80)
print("ALL MINI-ABLATIONS COMPLETE")
print("=" * 80)


In [ ]:
# Save Smoke Test Results
output_file = OUTPUT_PATH / "mini_ablations_smoke_test_results.json"
with open(output_file, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results saved: {output_file}")

# Summary
print("\nSummary:")
for ablation_id, result in results_summary.items():
    status_icon = "OK" if result['status'] == "success" else "FAIL"
    print(f"  {status_icon} {result['name']}: {result['status']}")

# Completion status
completion = {
    "status": "complete",
    "n_configs_tested": len(MINI_ABLATIONS),
    "n_successful": sum(1 for r in results_summary.values() if r['status'] == 'success'),
    "n_failed": sum(1 for r in results_summary.values() if r['status'] == 'failed')
}

completion_file = OUTPUT_PATH / "completion_status.json"
with open(completion_file, 'w') as f:
    json.dump(completion, f, indent=2)

print(f"\nCompletion status: {completion_file}")
print(f"Successful: {completion['n_successful']}/{completion['n_configs_tested']}")
